# One Look · 04 · Essai « 12 bandes » (Colab, GPU)

**Hypothèse :** donner au réseau les 12 bandes Sentinel-2 L2A, au lieu des 6 choisies à la main, et le laisser décider lesquelles servent. Une convolution 1 × 1 en entrée (le « mélange ») apprend à combiner les bandes ; une pénalité L1 la pousse à n'en garder que peu, ce qui fait apparaître les bandes utiles.

Deux entraînements, comparés au modèle de référence à 6 bandes (`mobilenet_v3_small_512_v3`), même réglage (512 px, 15 époques, étiquettes v3) :

| Run | Entrée |
|---|---|
| `mobilenet_v3_small_512_12b_v3` | les 12 bandes directement dans la première convolution |
| `mobilenet_v3_small_512_12b_melange_l1_v3` | mélange 1 × 1 (12 → 6 canaux), pénalité L1 = 0,01 |

**Avant de lancer :** GPU T4 ; secret Colab `HF_TOKEN` en **écriture**. Durée indicative : 10 min de préparation (paquet de 14 Go), environ 15 min par entraînement, 5 min d'évaluation.

In [ ]:
HF_DATA = "Elfried9684/onelook-alert-data"
HF_RUNS = "Elfried9684/onelook-alert-runs"
REPO_URL = "https://github.com/elfried96/onelook-alert_Elf.git"
WORKDIR = "/content/onelook-alert_Elf"
TAG = "v3"  # version des étiquettes
SIZE, EPOCHS = 512, 15
RUNS = [  # (suffixe du nom, options de scripts/train.py)
    ("12b", ""),
    ("12b_melange_l1", "--melange --l1 0.01"),
]

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : changer le type d'exécution"

## 1. Code, environnement et paquet à 12 bandes

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not os.path.exists(WORKDIR):
    !git clone -q {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} pull -q
%cd {WORKDIR}
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv sync --frozen --group train
!git log --oneline -1
!uv run hf download {HF_DATA} --repo-type dataset --include "pack12/*" --local-dir data --quiet
!uv run python scripts/check_pack.py --pack data/pack12

Essai éclair (deux pas sur 24 tuiles) : vérifie que le mélange et l'export ONNX à 12 bandes fonctionnent avant les vrais entraînements.

In [ ]:
!uv run python scripts/train.py --pack data/pack12 --size {SIZE} --melange --l1 0.01 --smoke --name essai_12b

## 2. Entraînements

In [ ]:
for suffix, options in RUNS:
    name = f"mobilenet_v3_small_{SIZE}_{suffix}_{TAG}"
    !uv run python scripts/train.py --pack data/pack12 --size {SIZE} --epochs {EPOCHS} --name {name} {options}

## 3. Quelles bandes le mélange a-t-il gardées ?

Poids de chaque bande d'entrée dans le mélange (somme des valeurs absolues sur les 6 canaux), au fil des époques.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
name = f"mobilenet_v3_small_{SIZE}_12b_melange_l1_{TAG}"
history = json.load(open(f"runs/{name}/history.json"))
weights = pd.DataFrame([h["melange"] for h in history], index=[h["epoque"] for h in history])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
weights.iloc[-1].sort_values().plot.barh(ax=ax[0], color="#2a78d6")
ax[0].set_title("Poids des bandes, dernière époque")
weights.plot(ax=ax[1], legend=False)
ax[1].set_title("Au fil des époques"); ax[1].set_xlabel("époque")
plt.tight_layout(); plt.show()
print(json.load(open(f"runs/{name}/config.json"))["importance_melange"])

## 4. Évaluation sur le test (ONNX, CPU)

In [ ]:
for suffix, _ in RUNS:
    !uv run python scripts/evaluate.py runs/mobilenet_v3_small_{SIZE}_{suffix}_{TAG} --pack data/pack12 --split test

## 5. Ranger les résultats

Dépôt **privé** Hugging Face, comme pour le notebook 03 (l'essai éclair n'est pas envoyé).

In [ ]:
!rm -rf runs/essai_12b
!uv run hf repos create {HF_RUNS} --type model --private --exist-ok
!uv run hf upload {HF_RUNS} runs runs --quiet
print(f"https://huggingface.co/{HF_RUNS}/tree/main/runs")

Ensuite, en local : `uv run hf download Elfried9684/onelook-alert-runs --include "runs/*12b*" --local-dir .`